# Assignment 1: Dataset Creation for N-gram Language Model

**Course:** GenAI for Software Development (CSCI 455/555)  
**Date:** February 2026  
**Instructor:** Dr. Antonio Mastropaolo  
**TA:** Md. Zahidul Haque Alvi

---

## Objective

In this notebook, we create a dataset of **tokenized Java methods** for training N-gram language models on code completion.

### Target Dataset Size
| Split | Size |
|-------|------|
| Training | ~20,000 methods |
| Validation | ~1,000 methods |
| Test | ~1,000 methods |

### Pipeline Overview
1. Clone repositories from GitHub
2. Select Java files (20 classes per repo)
3. Parse and extract methods
4. Filter methods (non-ASCII, minimum 10 tokens)
5. Tokenize (space-separated)
6. Clean (remove incomplete methods, multi-method lines)
7. Deduplicate (remove exact duplicates across dataset)
8. Split into train/val/test
9. Export dataset files

### Output Format
One tokenized method per line:
```
public void setName ( String name ) { this . name = name ; }
public int getAge ( ) { return this . age ; }
```

---
## Cell 1: Setup & Imports

In [ ]:
# Install required packages
!pip install javalang gitpython pandas

In [ ]:
import os
import glob
import subprocess
import json
import random
import shutil
from pathlib import Path

import pandas as pd
import javalang
from javalang.tokenizer import tokenize

# Configuration
CLONE_DIR = "/content/drive/MyDrive/GenAI/Assignment_1/dataset/java_repos"
OUTPUT_DIR = "/content/drive/MyDrive/GenAI/Assignment_1/dataset/ngram_dataset"

CLASSES_PER_REPO = 20   # Java files to sample per repo
MIN_TOKENS = 10         # Minimum tokens per method

VAL_SIZE = 1000
TEST_SIZE = 1000

# Create directories
os.makedirs(CLONE_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("Setup complete!")
print(f"Clone directory: {CLONE_DIR}")
print(f"Output directory: {OUTPUT_DIR}")

Setup complete!
Clone directory: /content/drive/MyDrive/GenAI/Assignment_1/dataset/java_repos
Output directory: /content/drive/MyDrive/GenAI/Assignment_1/dataset/ngram_dataset


---
## Fetch Repository List

We fetch top Java repositories from GitHub API with the following filters:

- **Language:** Java
- **Stars:** >1000 (to ensure high-quality, well-maintained projects)
- **Forks:** excluded (to avoid duplicate code)

We exclude forked repositories to avoid code duplication, as done in [Tufano et al., 2024](https://arxiv.org/pdf/2402.16480). We use a higher star threshold (>1000 vs >10 in that paper) since our goal is to create quality training data, not to maximize repository coverage.

In [ ]:
import requests

def fetch_top_java_repos(num_repos=200, per_page=100):
    """
    Fetch top-starred Java repositories from GitHub API.
    Skips forked repos to avoid duplicate code.
    """
    repos = []
    page = 1

    while len(repos) < num_repos:
        url = "https://api.github.com/search/repositories"
        params = {
            "q": "language:java stars:>1000",
            "sort": "stars",
            "order": "desc",
            "per_page": per_page,
            "page": page
        }

        response = requests.get(url, params=params)

        if response.status_code != 200:
            print(f"Error: {response.status_code}")
            break

        data = response.json()
        items = data.get("items", [])

        if not items:
            break

        for item in items:
            if item.get("fork", False):
                continue

            repos.append({
                "full_name": item["full_name"],
                "clone_url": item["clone_url"],
                "stars": item["stargazers_count"],
                "description": item.get("description", "")
            })

        page += 1

        if len(repos) >= num_repos:
            break

    return repos[:num_repos]

# Fetch repositories
print("Fetching top Java repositories from GitHub...")
repo_data = fetch_top_java_repos(num_repos=200)
df_repos = pd.DataFrame(repo_data)

print(f"\nFetched {len(df_repos)} repositories")
print(f"\nTop 10 repos by stars:")
df_repos.head(10)

Fetching top Java repositories from GitHub...

Fetched 200 repositories

Top 10 repos by stars:


,full_name,clone_url,stars,description
0,Snailclimb/JavaGuide,https://github.com/Snailclimb/JavaGuide.git,153693,Java 面试 & 后端通用面试指南，覆盖计算机基础、数据库、分布式、高并发与系统设计。准备...
1,krahets/hello-algo,https://github.com/krahets/hello-algo.git,122045,《Hello 算法》：动画图解、一键运行的数据结构与算法教程。支持简中、繁中、English...
2,GrowingGit/GitHub-Chinese-Top-Charts,https://github.com/GrowingGit/GitHub-Chinese-T...,105941,:cn: GitHub中文排行榜，各语言分设「软件 | 资料」榜单，精准定位中文好项目。各取...
3,iluwatar/java-design-patterns,https://github.com/iluwatar/java-design-patter...,93677,Design patterns implemented in Java
4,macrozheng/mall,https://github.com/macrozheng/mall.git,82834,mall项目是一套电商系统，包括前台商城系统及后台管理系统，基于Spring Boot+My...
5,spring-projects/spring-boot,https://github.com/spring-projects/spring-boot...,79818,Spring Boot helps you to create Spring-powered...
6,doocs/advanced-java,https://github.com/doocs/advanced-java.git,78805,😮 Core Interview Questions & Answers For Exper...
7,MisterBooo/LeetCodeAnimation,https://github.com/MisterBooo/LeetCodeAnimatio...,76705,Demonstrate all the questions on LeetCode in t...
8,elastic/elasticsearch,https://github.com/elastic/elasticsearch.git,76003,"Free and Open Source, Distributed, RESTful Sea..."
9,TheAlgorithms/Java,https://github.com/TheAlgorithms/Java.git,64914,All Algorithms implemented in Java


---
## Clone Repositories

We clone each repository using `--depth 1` (shallow clone) since we only need the current code snapshot, not the full commit history. This saves time and disk space.

In [ ]:
def clone_repo(clone_url, dest_dir):
    """
    Shallow clone a repository.
    Returns True if successful, False otherwise.
    """
    try:
        if os.path.exists(dest_dir):
            shutil.rmtree(dest_dir)

        cmd = ["git", "clone", "--depth", "1", "--quiet", clone_url, dest_dir]
        result = subprocess.run(cmd, capture_output=True, text=True, timeout=120)

        return result.returncode == 0
    except subprocess.TimeoutExpired:
        print(f"  Timeout cloning {clone_url}")
        return False
    except Exception as e:
        print(f"  Error: {e}")
        return False


# Clone repositories
cloned_repos = []
failed_repos = []

print(f"Cloning {len(df_repos)} repositories...\n")

for idx, row in df_repos.iterrows():
    repo_name = row["full_name"]
    clone_url = row["clone_url"]

    safe_name = repo_name.replace("/", "_")
    dest_dir = os.path.join(CLONE_DIR, safe_name)

    print(f"[{idx+1}/{len(df_repos)}] Cloning {repo_name}...", end=" ")

    success = clone_repo(clone_url, dest_dir)

    if success:
        cloned_repos.append({
            "repo_name": repo_name,
            "local_path": dest_dir,
            "stars": row["stars"]
        })
        print("done")
    else:
        failed_repos.append(repo_name)
        print("failed")

print(f"\n\nSummary:")
print(f"  Successfully cloned: {len(cloned_repos)}")
print(f"  Failed: {len(failed_repos)}")

Cloning 200 repositories...

[1/200] Cloning Snailclimb/JavaGuide... ✓
[2/200] Cloning krahets/hello-algo... ✓
[3/200] Cloning GrowingGit/GitHub-Chinese-Top-Charts... ✓
[4/200] Cloning iluwatar/java-design-patterns... ✓
[5/200] Cloning macrozheng/mall... ✓
[6/200] Cloning spring-projects/spring-boot...   Timeout cloning https://github.com/spring-projects/spring-boot.git
✗
[7/200] Cloning doocs/advanced-java... ✓
[8/200] Cloning MisterBooo/LeetCodeAnimation... ✓
[9/200] Cloning elastic/elasticsearch...   Timeout cloning https://github.com/elastic/elasticsearch.git
✗
[10/200] Cloning TheAlgorithms/Java... ✓
[11/200] Cloning kdn251/interviews... ✓
[12/200] Cloning NationalSecurityAgency/ghidra...   Timeout cloning https://github.com/NationalSecurityAgency/ghidra.git
✗
[13/200] Cloning spring-projects/spring-framework...   Timeout cloning https://github.com/spring-projects/spring-framework.git
✗
[14/200] Cloning google/guava... ✓
[15/200] Cloning termux/termux-app... ✓
[16/200] Cloning Rea

---
## Find and Select Java Files

For each cloned repository, we:
1. Find all `.java` files (excluding test/example directories)
2. Randomly select up to 20 files per repo
3. Track which files were used (so students can build test sets from remaining files)

In [ ]:
def find_java_files(repo_path):
    """
    Find all .java files in a repository.
    Excludes test files and common non-source directories.
    """
    java_files = []
    exclude_patterns = ["test", "tests", "example", "examples", "sample", "demo", "generated"]

    for root, dirs, files in os.walk(repo_path):
        root_lower = root.lower()
        if any(pattern in root_lower for pattern in exclude_patterns):
            continue

        for file in files:
            if file.endswith(".java"):
                java_files.append(os.path.join(root, file))

    return java_files


def select_java_files(java_files, max_files):
    """
    Randomly select up to max_files from the list.
    """
    if len(java_files) <= max_files:
        return java_files
    return random.sample(java_files, max_files)


# Find and select Java files from each repo
repo_java_files = {}
all_selected_files = []

print(f"Finding Java files (selecting up to {CLASSES_PER_REPO} per repo)...\n")

for repo_info in cloned_repos:
    repo_name = repo_info["repo_name"]
    repo_path = repo_info["local_path"]

    java_files = find_java_files(repo_path)

    if not java_files:
        print(f"  {repo_name}: No Java files found")
        continue

    selected = select_java_files(java_files, max_files=CLASSES_PER_REPO)

    repo_java_files[repo_name] = {
        "total_files": len(java_files),
        "selected_files": [os.path.relpath(f, repo_path) for f in selected],
        "remaining_files": len(java_files) - len(selected)
    }

    all_selected_files.extend([(repo_name, f) for f in selected])
    print(f"  {repo_name}: {len(selected)}/{len(java_files)} files selected")

print(f"\nTotal Java files selected: {len(all_selected_files)}")

Finding Java files (selecting up to 20 per repo)...

  Snailclimb/JavaGuide: 1/1 files selected
  krahets/hello-algo: 20/340 files selected
  GrowingGit/GitHub-Chinese-Top-Charts: 1/1 files selected
  iluwatar/java-design-patterns: 20/1233 files selected
  macrozheng/mall: 20/508 files selected
  doocs/advanced-java: 1/1 files selected
  MisterBooo/LeetCodeAnimation: 11/11 files selected
  TheAlgorithms/Java: 20/790 files selected
  kdn251/interviews: 20/516 files selected
  google/guava: 20/1270 files selected
  termux/termux-app: 20/174 files selected
  ReactiveX/RxJava: 20/914 files selected
  skylot/jadx: 20/1133 files selected
  jeecgboot/JeecgBoot: 20/786 files selected
  apache/dubbo: 20/2392 files selected
  PhilJay/MPAndroidChart: 20/158 files selected
  halo-dev/halo: 20/894 files selected
  alibaba/arthas: 20/755 files selected
  TeamNewPipe/NewPipe: 20/300 files selected
  geekxh/hello-algorithm: 20/71 files selected
  airbnb/lottie-android: 20/215 files selected
  YunaiV/r

---
## Parse and Extract Methods

For each Java file, we:
1. Parse the source code using `javalang`
2. Extract all method declarations
3. Store the method body along with metadata (repo, file, method name)

In [ ]:
def read_file_content(file_path):
    """Read file content with multiple encoding fallbacks."""
    encodings = ['utf-8', 'latin-1', 'cp1252']

    for encoding in encodings:
        try:
            with open(file_path, 'r', encoding=encoding) as f:
                return f.read()
        except UnicodeDecodeError:
            continue

    return None


def extract_method_source(source_code, method_node, lines):
    """Extract the source code of a method by counting braces."""
    try:
        start_line = method_node.position.line - 1

        brace_count = 0
        started = False
        end_line = start_line

        for i in range(start_line, len(lines)):
            line = lines[i]
            for char in line:
                if char == '{':
                    brace_count += 1
                    started = True
                elif char == '}':
                    brace_count -= 1

            if started and brace_count == 0:
                end_line = i
                break

        method_lines = lines[start_line:end_line + 1]
        return '\n'.join(method_lines)

    except Exception:
        return None


def extract_methods_from_file(file_path, repo_name):
    """Parse a Java file and extract all methods."""
    methods = []

    source_code = read_file_content(file_path)
    if source_code is None:
        return methods

    lines = source_code.split('\n')

    try:
        tree = javalang.parse.parse(source_code)

        for path, node in tree.filter(javalang.tree.MethodDeclaration):
            method_source = extract_method_source(source_code, node, lines)

            if method_source:
                methods.append({
                    "repo": repo_name,
                    "file": os.path.basename(file_path),
                    "method_name": node.name,
                    "source": method_source
                })

    except javalang.parser.JavaSyntaxError:
        pass
    except Exception:
        pass

    return methods


# Extract methods from all selected files
all_methods = []

print(f"Extracting methods from {len(all_selected_files)} files...\n")

for i, (repo_name, file_path) in enumerate(all_selected_files):
    if (i + 1) % 100 == 0:
        print(f"  Processed {i + 1}/{len(all_selected_files)} files...")

    methods = extract_methods_from_file(file_path, repo_name)
    all_methods.extend(methods)

print(f"\nTotal methods extracted: {len(all_methods)}")

Extracting methods from 2993 files...

  Processed 100/2993 files...
  Processed 200/2993 files...
  Processed 300/2993 files...
  Processed 400/2993 files...
  Processed 500/2993 files...
  Processed 600/2993 files...
  Processed 700/2993 files...
  Processed 800/2993 files...
  Processed 900/2993 files...
  Processed 1000/2993 files...
  Processed 1100/2993 files...
  Processed 1200/2993 files...
  Processed 1300/2993 files...
  Processed 1400/2993 files...
  Processed 1500/2993 files...
  Processed 1600/2993 files...
  Processed 1700/2993 files...
  Processed 1800/2993 files...
  Processed 1900/2993 files...
  Processed 2000/2993 files...
  Processed 2100/2993 files...
  Processed 2200/2993 files...
  Processed 2300/2993 files...
  Processed 2400/2993 files...
  Processed 2500/2993 files...
  Processed 2600/2993 files...
  Processed 2700/2993 files...
  Processed 2800/2993 files...
  Processed 2900/2993 files...


Total methods extracted: 25585


---
## Filter Methods

We filter out methods that:
1. Contain non-ASCII characters (ensures clean tokenization)
2. Have fewer than 10 tokens (too short to be meaningful)

In [ ]:
def contains_non_ascii(text):
    """Check if text contains non-ASCII characters."""
    try:
        text.encode('ascii')
        return False
    except UnicodeEncodeError:
        return True


def count_tokens(source_code):
    """Count the number of Java tokens in source code."""
    try:
        tokens = list(tokenize(source_code))
        return len(tokens)
    except:
        return 0


# TODO: Write your filtering functions here


# Apply filters
filtered_methods = []

stats = {
    "total": len(all_methods),
    "non_ascii_dropped": 0,
    "too_short_dropped": 0,
    "kept": 0
}

print(f"Filtering {len(all_methods)} methods...\n")

for method in all_methods:
    source = method["source"]

    if contains_non_ascii(source):
        stats["non_ascii_dropped"] += 1
        continue

    token_count = count_tokens(source)
    if token_count < MIN_TOKENS:
        stats["too_short_dropped"] += 1
        continue

    # TODO: Apply your filters here

    method["token_count"] = token_count
    filtered_methods.append(method)
    stats["kept"] += 1

print(f"Filtering Results:")
print(f"  Total methods:        {stats['total']}")
print(f"  Dropped (non-ASCII):  {stats['non_ascii_dropped']}")
print(f"  Dropped (< {MIN_TOKENS} tokens): {stats['too_short_dropped']}")
print(f"  -------------------------")
print(f"  Methods kept:         {stats['kept']}")

Filtering 25585 methods...

Filtering Results:
  Total methods:        25585
  Dropped (non-ASCII):  814
  Dropped (< 10 tokens): 1392
  -------------------------
  Methods kept:         23379


---
## Tokenize Methods

We tokenize each method into space-separated tokens using `javalang.tokenizer`.

Output format:
```
public void setName ( String name ) { this . name = name ; }
```

In [ ]:
def tokenize_method(source_code):
    """Tokenize Java source code into space-separated tokens."""
    try:
        tokens = list(tokenize(source_code))
        token_values = [token.value for token in tokens]
        return ' '.join(token_values)
    except:
        return None


# Tokenize all methods
tokenized_methods = []

print(f"Tokenizing {len(filtered_methods)} methods...\n")

for method in filtered_methods:
    tokenized = tokenize_method(method["source"])

    if tokenized:
        tokenized_methods.append({
            "repo": method["repo"],
            "file": method["file"],
            "method_name": method["method_name"],
            "tokenized_code": tokenized,
            "token_count": method["token_count"]
        })

print(f"Successfully tokenized: {len(tokenized_methods)} methods")

# Show example
print(f"\nExample tokenized method:")
if tokenized_methods:
    example = tokenized_methods[0]
    print(f"  Repo: {example['repo']}")
    print(f"  File: {example['file']}")
    print(f"  Method: {example['method_name']}")
    print(f"  Tokens ({example['token_count']}):")
    print(f"  {example['tokenized_code'][:200]}..." if len(example['tokenized_code']) > 200 else f"  {example['tokenized_code']}")

Tokenizing 23379 methods...

Successfully tokenized: 23379 methods

--- Example tokenized method ---
Repo: Snailclimb/JavaGuide
File: TranslateRepo.java
Method: printHeader
Tokens (53):
private static void printHeader ( ) { System . out . println ( "=" . repeat ( 70 ) ) ; System . out . println ( "Repository Documentation Translation Tool" ) ; System . out . println ( "=" . repeat ( ...


---
## Clean, Deduplicate, and Split

We clean the dataset by removing:
- Methods with multiple method signatures in one line
- Incomplete methods (not ending with `}`)

Then we remove exact duplicates and split into train/val/test sets.

In [ ]:
def is_clean_method(tokenized_code):
    """Check if method is clean (single method, complete)."""
    method_keywords = tokenized_code.count("public ") + tokenized_code.count("private ") + tokenized_code.count("protected ")
    if method_keywords > 1:
        return False
    if not tokenized_code.endswith("}"):
        return False
    return True

# Clean
print(f"Before cleaning: {len(tokenized_methods)}")
tokenized_methods = [m for m in tokenized_methods if is_clean_method(m['tokenized_code'])]
print(f"After cleaning: {len(tokenized_methods)}")

# Deduplicate
seen = set()
unique_methods = []
for m in tokenized_methods:
    if m['tokenized_code'] not in seen:
        seen.add(m['tokenized_code'])
        unique_methods.append(m)

print(f"After dedup: {len(unique_methods)}")
tokenized_methods = unique_methods

# Shuffle and split
random.seed(42)
random.shuffle(tokenized_methods)

val_size = VAL_SIZE
test_size = TEST_SIZE
train_size = len(tokenized_methods) - val_size - test_size

train_data = tokenized_methods[:train_size]
val_data = tokenized_methods[train_size:train_size + val_size]
test_data = tokenized_methods[train_size + val_size:train_size + val_size + test_size]

print(f"\nDataset Split:")
print(f"  Training:   {len(train_data)} methods")
print(f"  Validation: {len(val_data)} methods")
print(f"  Test:       {len(test_data)} methods")

Before cleaning: 21676
After cleaning: 21676
After dedup: 20258

Dataset Split:
  Training:   18258 methods
  Validation: 1000 methods
  Test:       1000 methods


---
## Save Outputs

We save:
1. `train.txt` - One tokenized method per line
2. `val.txt` - Validation set
3. `test.txt` - Test set
4. `metadata.json` - Tracks which repos/files were used (for students)

In [ ]:
def save_txt(data, filename):
    """Save tokenized methods to a text file (one per line)."""
    filepath = os.path.join(OUTPUT_DIR, filename)
    with open(filepath, 'w', encoding='utf-8', errors='replace') as f:
        for method in data:
            f.write(method['tokenized_code'] + '\n')
    return filepath


# Save train/val/test files
print("Saving dataset files...\n")

train_path = save_txt(train_data, "train.txt")
print(f"  Saved: {train_path}")

val_path = save_txt(val_data, "val.txt")
print(f"  Saved: {val_path}")

test_path = save_txt(test_data, "test.txt")
print(f"  Saved: {test_path}")

# Save metadata
metadata = {
    "description": "Metadata for N-gram dataset. Shows which files were used for training.",
    "instructions_for_students": [
        "To create a test set from the SAME distribution:",
        "  1. Use the same repositories listed below",
        "  2. Select Java files NOT in 'selected_files'",
        "  3. Extract methods using the same preprocessing",
        "",
        "To create a test set is likely to produce a DISTRIBUTION SHIFT:",
        "  1. Use different repositories (e.g., less popular by stars)",
        "  2. Compare model performance on both test sets"
    ],
    "dataset_stats": {
        "train_size": len(train_data),
        "val_size": len(val_data),
        "test_size": len(test_data),
        "total_repos": len(repo_java_files),
        "min_tokens": MIN_TOKENS
    },
    "repos_used": repo_java_files
}

metadata_path = os.path.join(OUTPUT_DIR, "metadata.json")
with open(metadata_path, 'w', encoding='utf-8') as f:
    json.dump(metadata, f, indent=2)
print(f"  Saved: {metadata_path}")

print(f"\nAll files saved to: {OUTPUT_DIR}/")

Saving dataset files...

  ✓ Saved: /content/drive/MyDrive/GenAI/Assignment_1/dataset/ngram_dataset/train.txt
  ✓ Saved: /content/drive/MyDrive/GenAI/Assignment_1/dataset/ngram_dataset/val.txt
  ✓ Saved: /content/drive/MyDrive/GenAI/Assignment_1/dataset/ngram_dataset/test.txt
  ✓ Saved: /content/drive/MyDrive/GenAI/Assignment_1/dataset/ngram_dataset/metadata.json


All files saved to: /content/drive/MyDrive/GenAI/Assignment_1/dataset/ngram_dataset/


---
## Summary Statistics

In [ ]:
import statistics

all_token_counts = [m['token_count'] for m in tokenized_methods]

print("=" * 50)
print("         DATASET CREATION SUMMARY")
print("=" * 50)

print(f"\nRepositories:")
print(f"   Cloned:    {len(cloned_repos)}")
print(f"   Failed:    {len(failed_repos)}")

print(f"\nJava Files:")
print(f"   Selected:  {len(all_selected_files)}")

print(f"\nMethods:")
print(f"   Extracted: {stats['total']}")
print(f"   Filtered:  {stats['total'] - stats['kept']} removed")
print(f"   Final:     {len(tokenized_methods)}")

print(f"\nToken Statistics:")
print(f"   Min:    {min(all_token_counts)}")
print(f"   Max:    {max(all_token_counts)}")
print(f"   Mean:   {statistics.mean(all_token_counts):.1f}")
print(f"   Median: {statistics.median(all_token_counts):.1f}")

print(f"\nDataset Splits:")
print(f"   Train:      {len(train_data):,} methods")
print(f"   Validation: {len(val_data):,} methods")
print(f"   Test:       {len(test_data):,} methods")

print(f"\nOutput Files:")
print(f"   {OUTPUT_DIR}/train.txt")
print(f"   {OUTPUT_DIR}/val.txt")
print(f"   {OUTPUT_DIR}/test.txt")
print(f"   {OUTPUT_DIR}/metadata.json")

print("\n" + "=" * 50)
print("         DATASET READY FOR N-GRAM TRAINING")
print("=" * 50)

         DATASET CREATION SUMMARY

Repositories:
   Cloned:    167
   Failed:    33

Java Files:
   Selected:  2993

Methods:
   Extracted: 25585
   Filtered:  2206 removed
   Final:     20258

Token Statistics:
   Min:    10
   Max:    38644
   Mean:   57.8
   Median: 27.0

Dataset Splits:
   Train:      18,258 methods
   Validation: 1,000 methods
   Test:       1,000 methods

Output Files:
   /content/drive/MyDrive/GenAI/Assignment_1/dataset/ngram_dataset/train.txt
   /content/drive/MyDrive/GenAI/Assignment_1/dataset/ngram_dataset/val.txt
   /content/drive/MyDrive/GenAI/Assignment_1/dataset/ngram_dataset/test.txt
   /content/drive/MyDrive/GenAI/Assignment_1/dataset/ngram_dataset/metadata.json

         DATASET READY FOR N-GRAM TRAINING


---
## Summary

This notebook created a dataset for N-gram language model training by:

1. Fetching 200 top Java repositories from GitHub (stars >1000, no forks)
2. Selecting 20 Java files per repository (excluding test/example directories)
3. Extracting methods using `javalang` parser
4. Filtering out methods with non-ASCII characters or fewer than 10 tokens
5. Cleaning malformed entries (multiple methods per line, incomplete methods)
6. Removing duplicate methods across the dataset
7. Splitting into train/val/test sets

### Output Files
- `train.txt` - ~18K tokenized methods
- `val.txt` - 1K tokenized methods
- `test.txt` - 1K tokenized methods
- `metadata.json` - Tracks repos and files used (for future test set creation)